# Projet 7 - Augmented Alzheimer Dataset Classical ML Pipeline (SVM / KNN / Random Forest)

Pipeline: loading -> resize -> HSV -> HOG -> HSV hist -> LBP -> concat -> standardization -> PCA -> SVM/KNN/RF -> evaluation.
Dataset: `Dataset/AugmentedAlzheimerDataset` (4 classes), custom split with `train_test_split`, SEED=42.


In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn scikit-image scipy pywavelets tqdm pillow


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
)
from sklearn.decomposition import PCA

from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.filters import gaussian
from skimage import exposure
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
import pywt

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")



## 1) Dataset Loading + MRI-Oriented Preprocessing


In [ ]:
# Dataset-specific config
DATASET_DIR = Path("Dataset/AugmentedAlzheimerDataset")
TARGET_SIZE = (128, 128)
TEST_SIZE = 0.2
MAX_SAMPLES_PER_CLASS = None  # Example: 3000 for faster experimentation
USE_CLAHE = True

def crop_black_background(img_gray, threshold=0.05):
    mask = img_gray > threshold
    if not np.any(mask):
        return img_gray
    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return img_gray[y0:y1, x0:x1]

def preprocess_mri_image(img, target_size=(128, 128), use_clahe=True):
    if img.ndim == 3:
        img = rgb2gray(img)
    img = img.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0

    img = crop_black_background(img, threshold=0.05)
    img = resize(img, target_size, anti_aliasing=True, preserve_range=True).astype(np.float32)
    img = gaussian(img, sigma=0.5, preserve_range=True).astype(np.float32)

    # Normalize to [0, 1]
    img_min, img_max = img.min(), img.max()
    img = (img - img_min) / (img_max - img_min + 1e-9)

    # Optional contrast enhancement
    if use_clahe:
        img = exposure.equalize_adapthist(img, clip_limit=0.02)
        img = np.clip(img, 0, 1)

    return img.astype(np.float32)

def load_alzheimer_dataset(dataset_dir, target_size=(128, 128), max_samples_per_class=None):
    class_names_local = sorted([d.name for d in dataset_dir.iterdir() if d.is_dir()])
    X_list, y_list = [], []
    rng = np.random.default_rng(SEED)

    for class_idx, class_name in enumerate(class_names_local):
        class_dir = dataset_dir / class_name
        paths = [p for p in class_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
        if max_samples_per_class is not None and len(paths) > max_samples_per_class:
            chosen = rng.choice(len(paths), size=max_samples_per_class, replace=False)
            paths = [paths[i] for i in chosen]

        for img_path in tqdm(paths, desc=f"Loading {class_name}"):
            try:
                img = imread(img_path)
                img_pp = preprocess_mri_image(img, target_size=target_size, use_clahe=USE_CLAHE)
                X_list.append(img_pp)
                y_list.append(class_idx)
            except Exception:
                continue

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y, class_names_local

X_all, y_all, class_names = load_alzheimer_dataset(
    DATASET_DIR,
    target_size=TARGET_SIZE,
    max_samples_per_class=MAX_SAMPLES_PER_CLASS
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_all
)

print("Dataset path :", DATASET_DIR)
print("Image size   :", TARGET_SIZE)
print("Train:", X_train_raw.shape, y_train.shape)
print("Test :", X_test_raw.shape, y_test.shape)
print("Classes:", len(class_names), class_names)
print("Train counts:", np.bincount(y_train, minlength=len(class_names)))
print("Test counts :", np.bincount(y_test, minlength=len(class_names)))

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
idx = np.random.choice(len(X_train_raw), size=min(12, len(X_train_raw)), replace=False)
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X_train_raw[i], cmap="gray")
    ax.set_title(class_names[y_train[i]], fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(idx):]:
    ax.axis("off")
plt.suptitle("Preprocessed MRI samples (grayscale/ROI-cleaned)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()



## 2) Feature Branch A - First-Order Statistics


In [ ]:
def first_order_features(img):
    vals = img.ravel()
    hist, _ = np.histogram(vals, bins=64, range=(0, 1), density=True)
    hist = hist + 1e-12
    hist = hist / hist.sum()
    entropy = -(hist * np.log2(hist)).sum()
    p10, p25, p50, p75, p90 = np.percentile(vals, [10, 25, 50, 75, 90])
    return np.array([
        vals.mean(),
        vals.std(),
        vals.var(),
        entropy,
        skew(vals),
        kurtosis(vals),
        p10, p25, p50, p75, p90
    ], dtype=np.float32)

X_train_stats = np.vstack([first_order_features(x) for x in tqdm(X_train_raw, desc="Stats train")])
X_test_stats = np.vstack([first_order_features(x) for x in tqdm(X_test_raw, desc="Stats test")])
print("Branch A shapes:", X_train_stats.shape, X_test_stats.shape)



## 3) Feature Branch B - Texture (GLCM + LBP)


In [ ]:
def texture_features(img):
    img_u8 = (img * 255).astype(np.uint8)

    # GLCM
    distances = [1, 2, 4]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(img_u8, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
    glcm_feats = []
    for p_name in props:
        glcm_feats.append(graycoprops(glcm, p_name).mean())

    # LBP histogram
    P, R = 16, 2
    lbp = local_binary_pattern(img, P=P, R=R, method="uniform")
    n_bins = P + 2
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), density=True)
    lbp_hist = lbp_hist.astype(np.float32)

    return np.hstack([np.array(glcm_feats, dtype=np.float32), lbp_hist]).astype(np.float32)

X_train_tex = np.vstack([texture_features(x) for x in tqdm(X_train_raw, desc="Texture train")])
X_test_tex = np.vstack([texture_features(x) for x in tqdm(X_test_raw, desc="Texture test")])
print("Branch B shapes:", X_train_tex.shape, X_test_tex.shape)



## 4) Feature Branch C - Wavelet (DWT)


In [ ]:
def band_stats(band):
    vals = band.ravel().astype(np.float32)
    energy = np.mean(vals ** 2)
    hist, _ = np.histogram(vals, bins=64, density=True)
    hist = hist + 1e-12
    hist = hist / hist.sum()
    ent = -(hist * np.log2(hist)).sum()
    return [vals.mean(), vals.std(), energy, ent]

def wavelet_features(img, wavelet="db2"):
    LL, (LH, HL, HH) = pywt.dwt2(img, wavelet=wavelet)
    feats = []
    for b in [LL, LH, HL, HH]:
        feats.extend(band_stats(b))
    return np.array(feats, dtype=np.float32)

X_train_wav = np.vstack([wavelet_features(x) for x in tqdm(X_train_raw, desc="Wavelet train")])
X_test_wav = np.vstack([wavelet_features(x) for x in tqdm(X_test_raw, desc="Wavelet test")])
print("Branch C shapes:", X_train_wav.shape, X_test_wav.shape)



## 5) Feature Branch D - HOG


In [ ]:
def hog_features(img):
    return hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    ).astype(np.float32)

X_train_hog = np.vstack([hog_features(x) for x in tqdm(X_train_raw, desc="HOG train")])
X_test_hog = np.vstack([hog_features(x) for x in tqdm(X_test_raw, desc="HOG test")])
print("Branch D shapes:", X_train_hog.shape, X_test_hog.shape)



## 6) Feature Fusion


In [ ]:
X_train_feat = np.hstack([X_train_stats, X_train_tex, X_train_wav, X_train_hog]).astype(np.float32)
X_test_feat = np.hstack([X_test_stats, X_test_tex, X_test_wav, X_test_hog]).astype(np.float32)
print("Fused shape:", X_train_feat.shape, X_test_feat.shape)



## 7) Feature Space Cleaning (constant + correlation) and Scaling


In [ ]:
# Remove constant features
vt = VarianceThreshold(threshold=0.0)
X_train_vt = vt.fit_transform(X_train_feat)
X_test_vt = vt.transform(X_test_feat)
print("After variance threshold:", X_train_vt.shape[1], "features")

# Remove highly correlated features
corr = pd.DataFrame(X_train_vt).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
keep_mask = np.ones(X_train_vt.shape[1], dtype=bool)
if len(to_drop) > 0:
    keep_mask[np.array(to_drop, dtype=int)] = False
X_train_clean = X_train_vt[:, keep_mask]
X_test_clean = X_test_vt[:, keep_mask]
print("After correlation filter:", X_train_clean.shape[1], "features")

# Scaling (required for SVM/KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)



## 8) Feature Selection + PCA Variants


In [ ]:
# Stage A: Univariate filtering (ANOVA F-score; switch to mutual_info if needed)
k_a = min(1200, X_train_scaled.shape[1])
selector_a = SelectKBest(score_func=f_classif, k=k_a)
X_train_a = selector_a.fit_transform(X_train_scaled, y_train)
X_test_a = selector_a.transform(X_test_scaled)
print("After Stage A (ANOVA):", X_train_a.shape[1], "features")

# Stage B: Embedded ranking (Random Forest importance)
rf_ranker = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
rf_ranker.fit(X_train_a, y_train)
importances = rf_ranker.feature_importances_
k_b = min(400, X_train_a.shape[1])
top_idx = np.argsort(importances)[::-1][:k_b]
X_train_sel = X_train_a[:, top_idx]
X_test_sel = X_test_a[:, top_idx]
print("After Stage B (RF importance):", X_train_sel.shape[1], "features")

# PCA variants
pca95 = PCA(n_components=0.95, random_state=SEED)
X_train_pca95 = pca95.fit_transform(X_train_sel)
X_test_pca95 = pca95.transform(X_test_sel)
print("PCA95 shape:", X_train_pca95.shape, X_test_pca95.shape)

pca_fixed_n = min(120, X_train_sel.shape[1], X_train_sel.shape[0] - 1)
pca_fixed = PCA(n_components=pca_fixed_n, random_state=SEED)
X_train_pca_fixed = pca_fixed.fit_transform(X_train_sel)
X_test_pca_fixed = pca_fixed.transform(X_test_sel)
print("PCA fixed shape:", X_train_pca_fixed.shape, X_test_pca_fixed.shape)

# Optional explained variance plot
cum = np.cumsum(pca95.explained_variance_ratio_)
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(cum) + 1), cum, lw=2)
plt.axhline(0.95, color="r", ls="--")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA (95% variance) - cumulative curve")
plt.tight_layout()
plt.show()



## 9) SVM Training (linear + RBF on multiple feature views)


In [ ]:
svm_feature_sets = {
    "selected_no_pca": (X_train_sel, X_test_sel),
    "pca95": (X_train_pca95, X_test_pca95),
    "pca_fixed": (X_train_pca_fixed, X_test_pca_fixed),
}

svm_grid = {
    "kernel": ["linear", "rbf"],
    "C": [0.1, 1, 10],
    "gamma": ["scale"]
}

best_svm = None
best_svm_score = -np.inf
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

for fs_name, (Xtr, Xte) in svm_feature_sets.items():
    gs = GridSearchCV(
        SVC(random_state=SEED),
        param_grid=svm_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=0
    )
    t0 = time.time()
    gs.fit(Xtr, y_train)
    fit_time = time.time() - t0
    y_pred = gs.best_estimator_.predict(Xte)
    f1m = f1_score(y_test, y_pred, average="macro")
    print(f"[SVM][{fs_name}] best={gs.best_params_} | CV={gs.best_score_:.4f} | test_f1_macro={f1m:.4f}")
    if f1m > best_svm_score:
        best_svm_score = f1m
        best_svm = (fs_name, gs.best_estimator_, fit_time)

svm_best_set, svm_final, svm_fit_time = best_svm
X_train_svm, X_test_svm = svm_feature_sets[svm_best_set]
t0 = time.time()
y_pred_svm = svm_final.predict(X_test_svm)
svm_pred_time = time.time() - t0

print(f"Chosen SVM set: {svm_best_set}")
print(classification_report(y_test, y_pred_svm, target_names=class_names, digits=4))



## 10) KNN Training (k, metric, weights)


In [ ]:
knn_feature_sets = {
    "selected_no_pca": (X_train_sel, X_test_sel),
    "pca95": (X_train_pca95, X_test_pca95),
    "pca_fixed": (X_train_pca_fixed, X_test_pca_fixed),
}

knn_grid = {
    "n_neighbors": [3, 5, 7, 9],
    "metric": ["euclidean", "manhattan"],
    "weights": ["uniform", "distance"],
}

best_knn = None
best_knn_score = -np.inf
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

for fs_name, (Xtr, Xte) in knn_feature_sets.items():
    gs = GridSearchCV(
        KNeighborsClassifier(),
        param_grid=knn_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    )
    t0 = time.time()
    gs.fit(Xtr, y_train)
    fit_time = time.time() - t0
    y_pred = gs.best_estimator_.predict(Xte)
    f1m = f1_score(y_test, y_pred, average="macro")
    print(f"[KNN][{fs_name}] best={gs.best_params_} | CV={gs.best_score_:.4f} | test_f1_macro={f1m:.4f}")
    if f1m > best_knn_score:
        best_knn_score = f1m
        best_knn = (fs_name, gs.best_estimator_, fit_time)

knn_best_set, knn_final, knn_fit_time = best_knn
X_train_knn, X_test_knn = knn_feature_sets[knn_best_set]
t0 = time.time()
y_pred_knn = knn_final.predict(X_test_knn)
knn_pred_time = time.time() - t0

print(f"Chosen KNN set: {knn_best_set}")
print(classification_report(y_test, y_pred_knn, target_names=class_names, digits=4))



## 11) Random Forest Training (selected features, optional PCA)


In [ ]:
rf_feature_sets = {
    "selected_no_pca": (X_train_sel, X_test_sel),
    "pca95": (X_train_pca95, X_test_pca95),
}

rf_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 20, 40],
    "max_features": ["sqrt", "log2"],
}

best_rf = None
best_rf_score = -np.inf
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

for fs_name, (Xtr, Xte) in rf_feature_sets.items():
    gs = GridSearchCV(
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        param_grid=rf_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    )
    t0 = time.time()
    gs.fit(Xtr, y_train)
    fit_time = time.time() - t0
    y_pred = gs.best_estimator_.predict(Xte)
    f1m = f1_score(y_test, y_pred, average="macro")
    print(f"[RF][{fs_name}] best={gs.best_params_} | CV={gs.best_score_:.4f} | test_f1_macro={f1m:.4f}")
    if f1m > best_rf_score:
        best_rf_score = f1m
        best_rf = (fs_name, gs.best_estimator_, fit_time)

rf_best_set, rf_final, rf_fit_time = best_rf
X_train_rf, X_test_rf = rf_feature_sets[rf_best_set]
t0 = time.time()
y_pred_rf = rf_final.predict(X_test_rf)
rf_pred_time = time.time() - t0

print(f"Chosen RF set: {rf_best_set}")
print(classification_report(y_test, y_pred_rf, target_names=class_names, digits=4))



## 12) Final Evaluation (all models)


In [ ]:
cms = {
    "SVM": confusion_matrix(y_test, y_pred_svm),
    "KNN": confusion_matrix(y_test, y_pred_knn),
    "RF": confusion_matrix(y_test, y_pred_rf),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, cm) in zip(axes, cms.items()):
    sns.heatmap(
        cm,
        ax=ax,
        cmap="Blues",
        fmt="d",
        annot=True,
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()



## 13) Comparative Analysis (Reference-style plots)

This section extends the baseline comparison with:
- normalized confusion matrices,
- macro and weighted metrics,
- per-class F1 heatmap,
- one-vs-rest ROC-AUC curves for the 4 Alzheimer classes,
- a compact overall ranking score.



In [ ]:
models = {
    "SVM": {
        "estimator": svm_final,
        "y_pred": y_pred_svm,
        "fit_t": svm_fit_time,
        "pred_t": svm_pred_time,
        "X_test": X_test_svm,
    },
    "KNN": {
        "estimator": knn_final,
        "y_pred": y_pred_knn,
        "fit_t": knn_fit_time,
        "pred_t": knn_pred_time,
        "X_test": X_test_knn,
    },
    "RF": {
        "estimator": rf_final,
        "y_pred": y_pred_rf,
        "fit_t": rf_fit_time,
        "pred_t": rf_pred_time,
        "X_test": X_test_rf,
    },
}

rows = []
for name, payload in models.items():
    y_pred = payload["y_pred"]
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 macro": f1_score(y_test, y_pred, average="macro"),
        "F1 weighted": f1_score(y_test, y_pred, average="weighted"),
        "Precision macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Fit time (s)": payload["fit_t"],
        "Predict time (s)": payload["pred_t"],
    })

results_df = pd.DataFrame(rows).set_index("Model").sort_values("F1 macro", ascending=False)
print(results_df.round(4))
results_df.to_csv("alzheimer_ml_classique_results.csv")
print("Saved: alzheimer_ml_classique_results.csv")

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
metric_cols = ["Accuracy", "F1 macro", "F1 weighted"]
for ax, col in zip(axes, metric_cols):
    bars = ax.bar(results_df.index, results_df[col], color=sns.color_palette("Set2", n_colors=len(results_df)))
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.bar_label(bars, fmt="%.3f")
axes[2].set_ylabel("Score")
plt.suptitle("Global Performance Comparison", y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(results_df.index))
w = 0.35
bp = axes[0].bar(x - w / 2, results_df["Precision macro"], w, label="Precision macro")
br = axes[0].bar(x + w / 2, results_df["Recall macro"], w, label="Recall macro")
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df.index)
axes[0].set_ylim(0, 1)
axes[0].set_title("Precision vs Recall (macro)")
axes[0].legend()
axes[0].bar_label(bp, fmt="%.3f")
axes[0].bar_label(br, fmt="%.3f")

cost_df = results_df[["Fit time (s)", "Predict time (s)"]]
cost_df.plot(kind="bar", ax=axes[1])
axes[1].set_title("Computational Cost Comparison")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

per_class_f1 = {}
for name, payload in models.items():
    rep = classification_report(y_test, payload["y_pred"], target_names=class_names, output_dict=True, zero_division=0)
    per_class_f1[name] = {cls: rep[cls]["f1-score"] for cls in class_names}

f1_class_df = pd.DataFrame(per_class_f1).T
plt.figure(figsize=(10, 4.5))
sns.heatmap(f1_class_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar=True)
plt.title("Per-class F1 Score Heatmap")
plt.xlabel("Class")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, payload) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, payload["y_pred"], normalize="true")
    sns.heatmap(
        cm, ax=ax, cmap="Blues", vmin=0, vmax=1, cbar=False,
        xticklabels=class_names, yticklabels=class_names
    )
    ax.set_title(f"{name} - Normalized CM")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

y_test_bin = label_binarize(y_test, classes=np.arange(len(class_names)))
plt.figure(figsize=(9, 7))
for name, payload in models.items():
    est = payload["estimator"]
    Xte = payload["X_test"]

    if hasattr(est, "predict_proba"):
        y_score = est.predict_proba(Xte)
    elif hasattr(est, "decision_function"):
        y_score = est.decision_function(Xte)
        if y_score.ndim == 1:
            y_score = y_score[:, None]
    else:
        continue

    auc_macro = roc_auc_score(y_test_bin, y_score, multi_class="ovr", average="macro")
    fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
    plt.plot(fpr_micro, tpr_micro, lw=2, label=f"{name} (macro AUC={auc_macro:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.title("One-vs-Rest ROC Curves (micro-average)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

norm_fit = (results_df["Fit time (s)"] - results_df["Fit time (s)"].min()) / (results_df["Fit time (s)"].max() - results_df["Fit time (s)"].min() + 1e-9)
norm_pred = (results_df["Predict time (s)"] - results_df["Predict time (s)"].min()) / (results_df["Predict time (s)"].max() - results_df["Predict time (s)"].min() + 1e-9)
results_df["Composite score"] = 0.5 * results_df["F1 macro"] + 0.3 * results_df["Accuracy"] - 0.1 * norm_fit - 0.1 * norm_pred

ranked_df = results_df.sort_values("Composite score", ascending=False)
print("\nComposite ranking (higher is better):")
print(ranked_df[["Accuracy", "F1 macro", "Fit time (s)", "Predict time (s)", "Composite score"]].round(4))

plt.figure(figsize=(8, 4.5))
bars = plt.bar(ranked_df.index, ranked_df["Composite score"], color=sns.color_palette("viridis", n_colors=len(ranked_df)))
plt.title("Overall Model Ranking (quality-speed trade-off)")
plt.ylabel("Composite score")
plt.bar_label(bars, fmt="%.3f")
plt.tight_layout()
plt.show()



## Experimental Study - Structured Protocol

Planned experiments:
1. Baseline (raw pixels)
2. Texture only (GLCM + LBP)
3. Wavelet only (DWT statistics)
4. Texture + Wavelet
5. Full fused (stats +